In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive/OULAD_processed')

In [ ]:
import pandas as pd
OUTPUT_PATH = '/content/drive/MyDrive/OULAD_processed/'

# ════════════════════════════════════════════════════════════
# MODELE CLUSTERING FINAL : K-MEANS + EMBEDDINGS FINAL
# + TEST STABILITÉ MULTI-SEED + P-CEA
# ════════════════════════════════════════════════════════════

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json

SAVE_PATH = OUTPUT_PATH + "dynamic_profiling_kmeans/"

# Dataset principal
final_df_wcdmacp = pd.read_csv(
    SAVE_PATH + "final_df_with_kmeans_profiles.csv"
)

# P-CEA
pcea_df = pd.read_csv(
    SAVE_PATH + "pcea_kmeans_results.csv"
)


# Embeddings checkpoints
with open(SAVE_PATH + "emb_norm_cp.pkl", "rb") as f:
    emb_norm_cp = pickle.load(f)

# Profils checkpoints
with open(SAVE_PATH + "profiles_cp.pkl", "rb") as f:
    profiles_cp = pickle.load(f)

# Profils complets
profiles_full = np.load(
    SAVE_PATH + "profiles_full.npy"
)

# Embeddings complets
emb_full_norm = np.load(
    SAVE_PATH + "emb_full_norm.npy"
)

# Métriques
with open(SAVE_PATH + "clustering_metrics.pkl", "rb") as f:
    metrics_saved = pickle.load(f)

metrics_full  = metrics_saved['full']
metrics_cp_km = metrics_saved['checkpoints']

# Label mappings
with open(SAVE_PATH + "cluster_label_mappings.json", "r") as f:
    mappings = json.load(f)

print("Données rechargées")

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# ── Constantes ────────────────────────────────────────────────
K_OPTIMAL   = 4
CHECKPOINTS = [3, 7, 12]
SEEDS       = [42, 123, 0, 7, 2024, 99, 256, 1337, 21, 500]

profile_colors = ['#16A34A', '#2563EB', '#D97706', '#DC2626']

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist' : '#D97706',
    'Mid-Engager'           : '#F59E0B',
    'At-Risk Engager'       : '#2563EB',
    'Resource Skimmer'      : '#DC2626',
}

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Mid-Engager',
    'Assessment Specialist',
    'Highly Engaged Learner',
]

PROFILE_RISK_ORDER = {
    'Highly Engaged Learner': 0,
    'Assessment Specialist': 1,
    'Mid-Engager': 2,
    'At-Risk Engager': 3,
    'Resource Skimmer': 4,
}

behavior_features = [
    'total_clicks',
    'clicks_content_engagement',
    'clicks_assessment_activity',
    'clicks_social_activity',
    'clicks_resource_activity',
    'clicks_specialized_activity',
    'mean_score',
    'submission_rate',
    'n_active_days',
]
behavior_features = [f for f in behavior_features
                     if f in final_df_wcdmacp.columns]


# ════════════════════════════════════════════════════════════
# FONCTIONS UTILITAIRES
# ════════════════════════════════════════════════════════════

def dunn_index(X, labels):
    unique_labels = np.unique(labels)
    centers = np.array([X[labels == k].mean(axis=0)
                        for k in unique_labels])
    inter_dists = [
        np.linalg.norm(centers[i] - centers[j])
        for i in range(len(unique_labels))
        for j in range(i+1, len(unique_labels))
    ]
    min_inter = min(inter_dists)
    max_intra = max(
        (cdist(X[labels == k], X[labels == k]).max()
         for k in unique_labels if (labels == k).sum() > 1),
        default=0
    )
    return min_inter / max_intra if max_intra > 0 else 0


def run_kmeans(embeddings, seed):
    """K-Means sur embeddings normalisés L2."""
    emb_norm = normalize(embeddings, norm='l2')
    km = KMeans(n_clusters=K_OPTIMAL, random_state=seed,
                n_init=10, max_iter=300)
    labels = km.fit_predict(emb_norm)
    sil    = silhouette_score(emb_norm, labels)
    dunn   = dunn_index(emb_norm, labels)
    db     = davies_bouldin_score(emb_norm, labels)
    return labels, km, emb_norm, {'sil': sil, 'dunn': dunn, 'db': db}


def normalize_heatmap(df):
    """Normalise chaque colonne entre 0 et 1."""
    df_norm = df.copy().astype(float)
    for col in df_norm.columns:
        cmin, cmax = df_norm[col].min(), df_norm[col].max()
        if cmax > cmin:
            df_norm[col] = (df_norm[col] - cmin) / (cmax - cmin)
    return df_norm


def plot_heatmap(data_norm, title, ylabel='Cluster'):
    fig, ax = plt.subplots(figsize=(14, max(4, len(data_norm))))
    sns.heatmap(data_norm, annot=True, fmt='.2f',
                cmap='YlOrRd', linewidths=0.5, ax=ax,
                cbar_kws={'label': '0=min, 1=max'})
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Variables comportementales')
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


def plot_risk_ladder(risk_df, cluster_col, title, mean_risk):
    fig, ax = plt.subplots(figsize=(9, 5))
    clusters = risk_df.index.tolist()
    risks    = risk_df['at_risk_rate'].tolist()
    colors   = [profile_colors[c % len(profile_colors)]
                if isinstance(c, int) else label_colors.get(c, '#94A3B8')
                for c in clusters]
    bars = ax.barh([str(c) for c in clusters],
                   [r*100 for r in risks],
                   color=colors, alpha=0.85)
    for bar, risk in zip(bars, risks):
        ax.text(bar.get_width() + 0.5,
                bar.get_y() + bar.get_height()/2,
                f'{risk*100:.1f}%',
                va='center', fontweight='bold')
    ax.axvline(x=mean_risk*100, color='black', ls='--',
               alpha=0.5, label=f'Moyenne ({mean_risk*100:.1f}%)')
    ax.set_xlabel('Taux at_risk (%)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 110)
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 1 — TEST STABILITE MULTI-SEED
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("ETAPE 1 — Stabilité multi-SEED")
print("=" * 60)

def test_stability(embeddings_dict, seeds):
    """
    Teste la stabilité du K-Means sur plusieurs seeds.
    embeddings_dict : {'full': emb, 3: emb_w3, 7: emb_w7, 12: emb_w12}
    Retourne le meilleur seed pour chaque horizon.
    """
    best_seeds = {}
    all_results = {}

    for name, emb in embeddings_dict.items():
        print(f"\n  {name}")
        print(f"  {'SEED':>6} | {'Sil':>8} | {'Dunn':>8} | {'DB':>8}")
        print(f"  {'-'*38}")

        results = []
        for seed in seeds:
            _, _, _, metrics = run_kmeans(emb, seed)
            results.append({'seed': seed, **metrics})
            print(f"  {seed:>6} | {metrics['sil']:>8.4f} | "
                  f"{metrics['dunn']:>8.4f} | {metrics['db']:>8.4f}")

        df = pd.DataFrame(results)
        all_results[name] = df
        best_seed = int(df.loc[df['sil'].idxmax(), 'seed'])
        best_seeds[name] = best_seed

        print(f"\n  Silhouette : {df['sil'].mean():.4f}"
              f" ± {df['sil'].std():.4f}")
        print(f"  → Meilleur SEED : {best_seed}")

    return best_seeds, all_results


embeddings_dict = {'full': embeddings_full}
for cp in CHECKPOINTS:
    embeddings_dict[cp] = embeddings_cp[cp]

best_seeds, stability_results = test_stability(
    embeddings_dict, SEEDS
)

# Graphique stabilité
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Stabilité K-Means — Silhouette par SEED',
             fontweight='bold', fontsize=13)

titles_stab = ['Séquence complète'] + [f'Checkpoint sem.{cp}'
                                        for cp in CHECKPOINTS]
names_stab  = ['full'] + CHECKPOINTS
colors_stab = ['#7C3AED', '#DC2626', '#D97706', '#2563EB']

for ax, name, title, color in zip(
    axes.flatten(), names_stab, titles_stab, colors_stab
):
    df = stability_results[name]
    ax.bar(df['seed'].astype(str), df['sil'],
           color=color, alpha=0.8)
    ax.axhline(y=df['sil'].mean(), color='red', ls='--', lw=1.5,
               label=f"Mean={df['sil'].mean():.4f}"
                     f"±{df['sil'].std():.4f}")
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('SEED')
    ax.set_ylabel('Silhouette')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 2 — CLUSTERING FINAL (meilleurs SEEDs)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ETAPE 2 — Clustering final")
print("=" * 60)

# Séquence complète
profiles_full, km_full, emb_full_norm, metrics_full = run_kmeans(
    embeddings_full, best_seeds['full']
)
print(f"\nSéquence complète (SEED={best_seeds['full']}) :"
      f" Sil={metrics_full['sil']:.4f}"
      f" | Dunn={metrics_full['dunn']:.4f}"
      f" | DB={metrics_full['db']:.4f}")

# Par checkpoint
profiles_cp   = {}
km_cp         = {}
emb_norm_cp   = {}
metrics_cp_km = {}

for cp in CHECKPOINTS:
    p, km, emb_n, m = run_kmeans(
        embeddings_cp[cp], best_seeds[cp]
    )
    profiles_cp[cp]   = p
    km_cp[cp]         = km
    emb_norm_cp[cp]   = emb_n
    metrics_cp_km[cp] = m
    print(f"Checkpoint sem.{cp} (SEED={best_seeds[cp]}) :"
          f" Sil={m['sil']:.4f}"
          f" | Dunn={m['dunn']:.4f}"
          f" | DB={m['db']:.4f}")

# Stocker dans final_df_wcdmacp
final_df_wcdmacp['cluster_full'] = profiles_full
for cp in CHECKPOINTS:
    final_df_wcdmacp[f'cluster_w{cp}'] = profiles_cp[cp]

print("\nClusters stockés dans final_df_wcdmacp ")

# Tableau métriques
metrics_summary = pd.DataFrame(
    [{'checkpoint': f'Sem. {cp}', **metrics_cp_km[cp]}
     for cp in CHECKPOINTS] +
    [{'checkpoint': 'Complet', **metrics_full}]
).set_index('checkpoint')
print("\nMétriques de validation :")
display(metrics_summary.round(4))

In [ ]:
# ════════════════════════════════════════════════════════════
# VISUALISATION t-SNE — CLUSTERS K-MEANS
# ════════════════════════════════════════════════════════════

from sklearn.manifold import TSNE

def plot_tsne_clusters(embeddings_norm, labels, title):
    print(f"Calcul t-SNE : {title}")

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate='auto',
        init='pca',
        random_state=SEED
    )

    emb_2d = tsne.fit_transform(embeddings_norm)

    plt.figure(figsize=(9, 7))

    for cid in range(K_OPTIMAL):
        mask = labels == cid
        plt.scatter(
            emb_2d[mask, 0],
            emb_2d[mask, 1],
            s=10,
            alpha=0.45,
            color=profile_colors[cid],
            label=f'Cluster {cid}'
        )

    plt.title(title, fontweight='bold')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    return emb_2d


# t-SNE séquence complète
emb_full_2d = plot_tsne_clusters(
    emb_full_norm,
    profiles_full,
    title=f't-SNE — Séquence complète | Sil={metrics_full["sil"]:.4f}'
)

# ════════════════════════════════════════════════════════════
# t-SNE CHECKPOINTS
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(18, 5))

for ax, cp in zip(axes, CHECKPOINTS):
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate='auto',
        init='pca',
        random_state=SEED
    )

    emb_2d = tsne.fit_transform(emb_norm_cp[cp])

    for cid in range(K_OPTIMAL):
        mask = profiles_cp[cp] == cid
        ax.scatter(
            emb_2d[mask, 0],
            emb_2d[mask, 1],
            s=8,
            alpha=0.45,
            color=profile_colors[cid],
            label=f'Cluster {cid}'
        )

    ax.set_title(
        f'Semaine {cp}\nSil={metrics_cp_km[cp]["sil"]:.4f}',
        fontweight='bold'
    )
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
plt.suptitle('t-SNE des embeddings GRU par checkpoint', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 3 — ANALYSE SÉQUENCE COMPLÈTE
# Heatmap + Trajectoires + Risk Ladder
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 3 — Analyse séquence complète")
print("=" * 60)

mean_risk = final_df_wcdmacp['at_risk'].mean()

# Profil moyen
cluster_means_full = final_df_wcdmacp.groupby('cluster_full')[
    behavior_features + ['at_risk']
].mean().round(3)
print("\nProfil moyen par cluster :")
display(cluster_means_full)

# Heatmap
plot_heatmap(
    normalize_heatmap(cluster_means_full[behavior_features]),
    'Signatures comportementales — Séquence complète\n'
    f'(SEED={best_seeds["full"]}, Sil={metrics_full["sil"]:.4f})'
)

# Risk Ladder
risk_ladder_full = final_df_wcdmacp.groupby('cluster_full').agg(
    n_students        = ('id_student',                 'count'),
    at_risk_rate      = ('at_risk',                    'mean'),
    mean_score        = ('mean_score',                 'mean'),
    total_clicks      = ('total_clicks',               'mean'),
    submission_rate   = ('submission_rate',            'mean'),
    assessment_clicks = ('clicks_assessment_activity', 'mean'),
    social_clicks     = ('clicks_social_activity',     'mean'),
).round(3).sort_values('at_risk_rate', ascending=False)

print("\nRisk Ladder — Séquence complète :")
display(risk_ladder_full)
plot_risk_ladder(risk_ladder_full, 'cluster_full',
                 'Risk Ladder — Séquence complète', mean_risk)

# Trajectoires temporelles
weekly_with_profile = weekly_combined.merge(
    final_df_wcdmacp[student_id_cols + ['cluster_full']],
    on=student_id_cols, how='left'
)
trajectory = weekly_with_profile.groupby(
    ['week', 'cluster_full']
)['weekly_clicks_capped'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Trajectoires temporelles — Séquence complète',
             fontweight='bold', fontsize=13)

for ax, (start, end, title) in zip(axes, [
    (0, 38, 'Trajectoires complètes (0→38)'),
    (0, 15, 'Zoom premières semaines (0→15)'),
]):
    data_traj = trajectory[trajectory['week'].between(start, end)]
    for cid in range(K_OPTIMAL):
        data = data_traj[data_traj['cluster_full'] == cid]
        ax.plot(data['week'], data['weekly_clicks_capped'],
                'o-', lw=2, ms=4, color=profile_colors[cid],
                label=f'Cluster {cid}')
    for cp in CHECKPOINTS:
        ax.axvline(x=cp, color='gray', ls=':', alpha=0.5, lw=1.5)
    ax.set_xlabel('Semaine')
    ax.set_ylabel('Clics moyens')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 4 — ANALYSE DES CHECKPOINTS
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 4 — Analyse des checkpoints")
print("=" * 60)

risk_checkpoints = {}

for cp in CHECKPOINTS:
    cluster_col = f'cluster_w{cp}'

    print(f"\n{'='*60}")
    print(f"CHECKPOINT — SEMAINE {cp} "
          f"(SEED={best_seeds[cp]} | "
          f"Sil={metrics_cp_km[cp]['sil']:.4f})")
    print('='*60)

    # Risk Ladder
    risk_cp = final_df_wcdmacp.groupby(cluster_col).agg(
        n_students        = ('id_student',                 'count'),
        at_risk_rate      = ('at_risk',                    'mean'),
        mean_score        = ('mean_score',                 'mean'),
        total_clicks      = ('total_clicks',               'mean'),
        submission_rate   = ('submission_rate',            'mean'),
        assessment_clicks = ('clicks_assessment_activity', 'mean'),
        social_clicks     = ('clicks_social_activity',     'mean'),
    ).round(3).sort_values('at_risk_rate', ascending=False)

    print(f"\nRisk Ladder — semaine {cp} :")
    display(risk_cp)
    plot_risk_ladder(risk_cp, cluster_col,
                     f'Risk Ladder — Semaine {cp}', mean_risk)

    # Heatmap
    cluster_means_cp = final_df_wcdmacp.groupby(
        cluster_col
    )[behavior_features].mean()
    plot_heatmap(
        normalize_heatmap(cluster_means_cp),
        f'Signatures comportementales — Semaine {cp}\n'
        f'(SEED={best_seeds[cp]}, '
        f'Sil={metrics_cp_km[cp]["sil"]:.4f})',
        ylabel=f'Cluster sem.{cp}'
    )

    risk_checkpoints[cp] = risk_cp

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — LABELLISATION MANUELLE
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 5 — Labellisation manuelle")
print("=" * 60)


# ══ À ADAPTER selon tes résultats ═══════════════════════════
CLUSTER_TO_LABEL_FINAL = {
    0: 'Resource Skimmer',
    1: 'Highly Engaged Learner',
    2: 'Assessment Specialist',
    3: 'At-Risk Engager'
}
CLUSTER_TO_LABEL_W3 = {
    0: 'Highly Engaged Learner',
    1: 'Resource Skimmer',
    2: 'At-Risk Engager',
    3: 'Mid-Engager',
}
CLUSTER_TO_LABEL_W7 = {
    0: 'Highly Engaged Learner',
    1: 'Resource Skimmer',
    2: 'At-Risk Engager',
    3: 'Assessment Specialist',
}
CLUSTER_TO_LABEL_W12 = {
    0: 'Resource Skimmer',
    1: 'Assessment Specialist',
    2: 'At-Risk Engager',
    3: 'Highly Engaged Learner',
}
# ════════════════════════════════════════════════════════════

checkpoint_mappings = {
    'full': CLUSTER_TO_LABEL_FINAL,
    3     : CLUSTER_TO_LABEL_W3,
    7     : CLUSTER_TO_LABEL_W7,
    12    : CLUSTER_TO_LABEL_W12,
}

# Appliquer les labels
final_df_wcdmacp['profile_label'] = (
    final_df_wcdmacp['cluster_full'].map(CLUSTER_TO_LABEL_FINAL)
)
for cp in CHECKPOINTS:
    final_df_wcdmacp[f'profile_label_w{cp}'] = (
        final_df_wcdmacp[f'cluster_w{cp}'].map(
            checkpoint_mappings[cp]
        )
    )

print("Labels appliqués :")
for cp in CHECKPOINTS:
    print(f"\n  Semaine {cp} :")
    print(final_df_wcdmacp[f'profile_label_w{cp}'].value_counts()
          .to_string())
print("\n  Final :")
print(final_df_wcdmacp['profile_label'].value_counts().to_string())

In [ ]:
print("\nVérification mapping final :")
display(
    final_df_wcdmacp.groupby(['cluster_full', 'profile_label']).size()
)

print("\nVérification comportement par label final :")
display(
    final_df_wcdmacp.groupby('profile_label')[
        ['total_clicks', 'mean_score', 'submission_rate', 'at_risk']
    ].mean().round(3)
)

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 6 — VALIDATION FINALE DES LABELS
# ════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("ÉTAPE 6 — Validation finale des labels")
print("=" * 60)

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Assessment Specialist',
    'Highly Engaged Learner'
]

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist': '#D97706',
    'At-Risk Engager': '#2563EB',
    'Resource Skimmer': '#DC2626',
}

# ── Heatmap finale avec labels ───────────────────────────────
cluster_means_labeled = final_df_wcdmacp.groupby(
    'profile_label'
)[behavior_features].mean()

cluster_means_labeled = cluster_means_labeled.reindex(profile_order)

plot_heatmap(
    normalize_heatmap(cluster_means_labeled),
    'Signatures comportementales — Labels finaux',
    ylabel='Profil'
)

# ── Risk Ladder final avec labels ────────────────────────────
risk_final = final_df_wcdmacp.groupby('profile_label').agg(
    n_students      = ('id_student',       'count'),
    at_risk_rate    = ('at_risk',          'mean'),
    mean_score      = ('mean_score',       'mean'),
    total_clicks    = ('total_clicks',     'mean'),
    submission_rate = ('submission_rate',  'mean'),
).round(3).sort_values('at_risk_rate', ascending=False)

print("\nRisk Ladder final :")
display(risk_final)
mean_risk = final_df_wcdmacp['at_risk'].mean()
plot_risk_ladder(
    risk_final,
    'profile_label',
    'Risk Ladder — Labels finaux',
    mean_risk
)

# ── Average Weekly Click Activity ────────────────────────────
# Ici on utilise total_clicks / 38 pour éviter le problème des semaines manquantes
# et des moyennes hebdomadaires trompeuses.

N_WEEKS = 38

final_df_wcdmacp['avg_weekly_click_activity'] = (
    final_df_wcdmacp['total_clicks'] / N_WEEKS
)

avg_weekly_profile = final_df_wcdmacp.groupby('profile_label').agg(
    n_students=('id_student', 'count'),
    avg_weekly_click_activity=('avg_weekly_click_activity', 'mean'),
    total_clicks=('total_clicks', 'mean'),
    at_risk_rate=('at_risk', 'mean')
).reindex(profile_order).round(3)

print("\nAverage Weekly Click Activity par profil :")
display(avg_weekly_profile)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=final_df_wcdmacp,
    x='profile_label',
    y='avg_weekly_click_activity',
    order=profile_order,
    palette=label_colors,
    errorbar=None
)

plt.title(
    'Average Weekly Click Activity par profil',
    fontweight='bold',
    fontsize=13
)
plt.xlabel('Profil')
plt.ylabel('Average Weekly Click Activity')
plt.xticks(rotation=25, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nValidation finale terminée")

In [ ]:
# ════════════════════════════════════════════════════════════
# TRAJECTOIRES TEMPORELLES PROPRES
# ════════════════════════════════════════════════════════════

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Mid-Engager',
    'Assessment Specialist',
    'Highly Engaged Learner'
]

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist': '#D97706',
    'Mid-Engager': '#F59E0B',
    'At-Risk Engager': '#2563EB',
    'Resource Skimmer': '#DC2626',
}

# reconstruire une grille complète étudiant × semaine
profile_df = final_df_wcdmacp[
    student_id_cols + ['profile_label']
].drop_duplicates()

weeks_df = pd.DataFrame({'week': range(0, 39)})

full_grid = profile_df.merge(weeks_df, how='cross')

weekly_tmp = weekly_combined[
    student_id_cols + ['week', 'weekly_clicks']
].copy()

weekly_full = full_grid.merge(
    weekly_tmp,
    on=student_id_cols + ['week'],
    how='left'
)

# semaine sans activité = 0 clic
weekly_full['weekly_clicks'] = weekly_full['weekly_clicks'].fillna(0)

# moyenne hebdomadaire par profil
trajectory_labeled = weekly_full.groupby(
    ['week', 'profile_label']
)['weekly_clicks'].mean().reset_index()

# lisser légèrement pour une courbe plus lisible
trajectory_labeled['weekly_clicks_smooth'] = (
    trajectory_labeled
    .sort_values(['profile_label', 'week'])
    .groupby('profile_label')['weekly_clicks']
    .transform(lambda s: s.rolling(window=3, center=True, min_periods=1).mean())
)

plt.figure(figsize=(14, 7))

for label in profile_order:
    data = trajectory_labeled[
        trajectory_labeled['profile_label'] == label
    ]

    plt.plot(
        data['week'],
        data['weekly_clicks_smooth'],
        marker='o',
        linewidth=2.5,
        markersize=5,
        color=label_colors[label],
        label=label
    )

milestones = {
    7: 'Month 1',
    14: 'Mid-term',
    21: 'Month 3',
    35: 'Finals'
}

for week, text in milestones.items():
    plt.axvline(
        x=week,
        color='gray',
        linestyle='--',
        alpha=0.5,
        linewidth=1.5
    )
    plt.text(
        week + 0.2,
        plt.ylim()[1] * 0.95,
        text,
        rotation=45,
        color='gray',
        fontsize=9,
        va='top'
    )

plt.title(
    'Temporal Click Activity by Student Archetype',
    fontweight='bold',
    fontsize=14
)
plt.xlabel('Week')
plt.ylabel('Average Weekly Click Activity')
plt.legend(title='Student Archetypes')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# VALIDATION DES LABELS — CHECKPOINTS
# ════════════════════════════════════════════════════════════

checkpoint_label_cols = {
    3: 'profile_label_w3',
    7: 'profile_label_w7',
    12: 'profile_label_w12',
}


for cp in CHECKPOINTS:

    print("\n" + "=" * 60)
    print(f"VALIDATION LABELS — CHECKPOINT SEMAINE {cp}")
    print("=" * 60)

    label_col = checkpoint_label_cols[cp]

    # ── Heatmap ──────────────────────────────────────────────
    cluster_cp = final_df_wcdmacp.groupby(
        label_col
    )[behavior_features].mean()

    present_profiles = [
        p for p in profile_order
        if p in final_df_wcdmacp[label_col].dropna().unique()
        ]

    cluster_cp = cluster_cp.reindex(present_profiles)

    plot_heatmap(
        normalize_heatmap(cluster_cp),
        f'Signatures comportementales — Semaine {cp}',
        ylabel='Profil'
    )

    # ── Risk Ladder ──────────────────────────────────────────
    risk_cp = final_df_wcdmacp.groupby(label_col).agg(
        n_students      = ('id_student',      'count'),
        at_risk_rate    = ('at_risk',         'mean'),
        mean_score      = ('mean_score',      'mean'),
        total_clicks    = ('total_clicks',    'mean'),
        submission_rate = ('submission_rate', 'mean'),
    ).round(3).sort_values(
        'at_risk_rate',
        ascending=False
    )

    print(f"\nRisk Ladder — semaine {cp}")
    display(risk_cp)

    plot_risk_ladder(
        risk_cp,
        label_col,
        f'Risk Ladder — Semaine {cp}',
        mean_risk
    )

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 7 — Profile Cluster Evolution Analysis
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 7 — P-CEA (Profile Cluster Evolution Analysis)")
print("=" * 60)


profile_steps = [f'profile_label_w{cp}' for cp in CHECKPOINTS] \
                + ['profile_label']
step_names    = [f'Semaine {cp}' for cp in CHECKPOINTS] + ['Final']

pcea_df = final_df_wcdmacp[
    student_id_cols + profile_steps + ['at_risk']
].copy()

# Niveaux de risque
for col in profile_steps:
    level_col = col.replace('profile_label', 'risk_level')
    pcea_df[level_col] = pcea_df[col].map(PROFILE_RISK_ORDER)

trajectory_cols = [col.replace('profile_label', 'risk_level')
                   for col in profile_steps]

# Calcul tendance
def compute_trend(row):
    vals = [row[c] for c in trajectory_cols
            if not pd.isna(row.get(c))]
    if len(vals) < 2:
        return np.nan
    return float(np.polyfit(range(len(vals)), vals, 1)[0])

pcea_df['profile_trend'] = pcea_df.apply(compute_trend, axis=1)
pcea_df['trajectory']    = pd.cut(
    pcea_df['profile_trend'],
    bins=[-np.inf, -0.2, 0.2, np.inf],
    labels=['Amélioration', 'Stable', 'Dégradation']
)
pcea_df['trajectory_num'] = pcea_df['trajectory'].map({
    'Amélioration': -1, 'Stable': 0, 'Dégradation': 1
})

# Supprimer anciennes colonnes P-CEA si le bloc a déjà été exécuté
cols_to_drop = [
    'profile_trend',
    'trajectory',
    'trajectory_num',
    'profile_trend_x',
    'trajectory_x',
    'trajectory_num_x',
    'profile_trend_y',
    'trajectory_y',
    'trajectory_num_y',
]

final_df_wcdmacp = final_df_wcdmacp.drop(
    columns=[c for c in cols_to_drop if c in final_df_wcdmacp.columns]
)

# Fusionner les nouveaux résultats P-CEA
final_df_wcdmacp = final_df_wcdmacp.merge(
    pcea_df[student_id_cols + [
        'profile_trend',
        'trajectory',
        'trajectory_num'
    ]],
    on=student_id_cols,
    how='left'
)

print("\nDistribution des trajectoires :")
print(final_df_wcdmacp['trajectory'].value_counts().to_string())

traj_risk = final_df_wcdmacp.groupby(
    'trajectory', observed=True
).agg(
    n_students  = ('at_risk', 'count'),
    at_risk_rate= ('at_risk', 'mean'),
    mean_trend  = ('profile_trend', 'mean')
).round(3)
print("\nRésumé P-CEA :")
display(traj_risk)


# ── Sankey P-CEA ─────────────────────────────────────────────
labels_sankey   = []
node_colors_sk  = []
for step in step_names:
    for profile in profile_order:
        labels_sankey.append(f"{step}<br>{profile}")
        node_colors_sk.append(label_colors[profile])

label_to_idx = {l: i for i, l in enumerate(labels_sankey)}
sources, targets, values_sk, link_colors_sk = [], [], [], []

for i in range(len(profile_steps) - 1):
    col_from = profile_steps[i]
    col_to   = profile_steps[i+1]
    transitions = pcea_df.groupby(
        [col_from, col_to]
    ).size().reset_index(name='count')

    for _, row in transitions.iterrows():
        src = f"{step_names[i]}<br>{row[col_from]}"
        tgt = f"{step_names[i+1]}<br>{row[col_to]}"
        if src in label_to_idx and tgt in label_to_idx:
            sources.append(label_to_idx[src])
            targets.append(label_to_idx[tgt])
            values_sk.append(row['count'])
            link_colors_sk.append('rgba(150,150,150,0.25)')

fig_sankey = go.Figure(data=[go.Sankey(
    arrangement = "snap",
    node = dict(
        pad=18, thickness=18,
        line=dict(color="black", width=0.3),
        label=labels_sankey, color=node_colors_sk
    ),
    link = dict(
        source=sources, target=targets,
        value=values_sk, color=link_colors_sk
    )
)])
fig_sankey.update_layout(
    title_text = "P-CEA — Migrations des profils — K-Means",
    font_size  = 11, height=650, width=1200
)
fig_sankey.show()


# ── Evolution nombre d'étudiants par profil ────────────────
evolution_counts = [
    {'step': sn, 'profile': p,
     'n_students': pcea_df[col].value_counts().get(p, 0)}
    for sn, col in zip(step_names, profile_steps)
    for p in profile_order
]
evolution_df = pd.DataFrame(evolution_counts)

plt.figure(figsize=(12, 6))
for profile in profile_order:
    data = evolution_df[evolution_df['profile'] == profile]
    plt.plot(data['step'], data['n_students'],
             marker='o', lw=2.5, label=profile,
             color=label_colors[profile])
    for _, row in data.iterrows():
        plt.text(row['step'], row['n_students'] + 200,
                 f"{int(row['n_students'])}", ha='center', fontsize=9)

plt.title("P-CEA — Évolution du nombre d'étudiants par profil",
          fontweight='bold')
plt.xlabel("Étape temporelle")
plt.ylabel("Nombre d'étudiants")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ── Matrices de transition ────────────────────────────────

transition_pairs = [
    (profile_steps[i], profile_steps[i+1],
     f'{step_names[i]} → {step_names[i+1]}')
    for i in range(len(profile_steps) - 1)
]

def get_profiles_present(col):
    return [
        p for p in profile_order
        if p in pcea_df[col].dropna().unique()
    ]

for col_from, col_to, title in transition_pairs:

    row_order = get_profiles_present(col_from)
    col_order = get_profiles_present(col_to)

    matrix = pd.crosstab(
        pcea_df[col_from],
        pcea_df[col_to],
        normalize='index'
    ).reindex(
        index=row_order,
        columns=col_order
    ).fillna(0)

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        matrix,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5,
        cbar_kws={'label': 'Probabilité de transition'}
    )

    plt.title(
        f"P-CEA — Matrice de transition\n{title}",
        fontweight='bold'
    )
    plt.xlabel("Profil destination")
    plt.ylabel("Profil origine")
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
OLD_PATH = OUTPUT_PATH + "dynamic_profiling_kmeans/"

with open(OLD_PATH + "kmeans_full.pkl", "rb") as f:
    km_full = pickle.load(f)

with open(OLD_PATH + "kmeans_checkpoints.pkl", "rb") as f:
    km_cp = pickle.load(f)

In [ ]:
with open(OUTPUT_PATH + "dynamic_profiling_kmeans/cluster_label_mappings.json") as f:
    old_mapping = json.load(f)

label_mappings_save['best_seeds'] = old_mapping.get('best_seeds', {})

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 8 — SAUVEGARDE COMPLETE
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ETAPE 8 — Sauvegarde")
print("=" * 60)

SAVE_PATH = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels/"
os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset final
final_df_wcdmacp.to_csv(
    SAVE_PATH + "final_df_with_kmeans_profiles.csv", index=False
)

# P-CEA
pcea_df.to_csv(SAVE_PATH + "pcea_kmeans_results.csv", index=False)

# Modèles K-Means
with open(SAVE_PATH + "kmeans_full.pkl", "wb") as f:
    pickle.dump(km_full, f)
with open(SAVE_PATH + "kmeans_checkpoints.pkl", "wb") as f:
    pickle.dump(km_cp, f)

# Label mappings
label_mappings_save = {
    str(k): {str(ki): vi for ki, vi in v.items()}
    for k, v in checkpoint_mappings.items()
}
label_mappings_save['risk_order'] = PROFILE_RISK_ORDER
label_mappings_save['best_seeds'] = {
    str(k): v for k, v in best_seeds.items()
}
with open(SAVE_PATH + "cluster_label_mappings.json", "w") as f:
    json.dump(label_mappings_save, f, indent=4)

# Métriques
with open(SAVE_PATH + "clustering_metrics.pkl", "wb") as f:
    pickle.dump({'full': metrics_full,
                 'checkpoints': metrics_cp_km}, f)

# Embeddings
np.save(SAVE_PATH + "emb_full_norm.npy",  emb_full_norm)
np.save(SAVE_PATH + "profiles_full.npy",  profiles_full)
with open(SAVE_PATH + "emb_norm_cp.pkl",  "wb") as f:
    pickle.dump(emb_norm_cp, f)
with open(SAVE_PATH + "profiles_cp.pkl",  "wb") as f:
    pickle.dump(profiles_cp, f)

# Stabilité
for name, df in stability_results.items():
    df.to_csv(SAVE_PATH + f"stability_{name}.csv", index=False)

# Résumés
risk_final.to_csv(SAVE_PATH + "risk_ladder_final.csv")
cluster_means_labeled.to_csv(SAVE_PATH + "cluster_means_labeled.csv")

print(" Sauvegarde terminée :")
for f in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024
    print(f"  {f:<45} ({size:.1f} KB)")

print("\n" + "=" * 60)
print(" DYNAMIC PROFILING TERMINÉ")
print("=" * 60)
print(f"\nMeilleurs SEEDs :")
for name, seed in best_seeds.items():
    print(f"  {str(name):<8} → SEED={seed}")
print(f"\nMétriques finales :")
print(f"  Complet | Sil={metrics_full['sil']:.4f} | "
      f"Dunn={metrics_full['dunn']:.4f} | DB={metrics_full['db']:.4f}")
for cp in CHECKPOINTS:
    m = metrics_cp_km[cp]
    print(f"  Sem.{cp:2d}  | Sil={m['sil']:.4f} | "
          f"Dunn={m['dunn']:.4f} | DB={m['db']:.4f}")
print(f"\n→ Prêt pour M4 (XAI — SHAP) ")

In [ ]:
label_mappings_save = {
    str(k): {str(ki): vi for ki, vi in v.items()}
    for k, v in checkpoint_mappings.items()
}

label_mappings_save['risk_order'] = PROFILE_RISK_ORDER

In [ ]:
with open(
    OUTPUT_PATH + "dynamic_profiling_kmeans/cluster_label_mappings.json",
    "r"
) as f:
    old_mapping = json.load(f)

label_mappings_save['best_seeds'] = old_mapping.get(
    'best_seeds', {}
)

In [ ]:
SAVE_PATH_FIXED = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels/"
os.makedirs(SAVE_PATH_FIXED, exist_ok=True)

final_df_wcdmacp.to_csv(
    SAVE_PATH_FIXED + "final_df_with_kmeans_profiles.csv",
    index=False
)

pcea_df.to_csv(
    SAVE_PATH_FIXED + "pcea_kmeans_results.csv",
    index=False
)

with open(SAVE_PATH_FIXED + "cluster_label_mappings.json", "w") as f:
    json.dump(label_mappings_save, f, indent=4)

risk_final.to_csv(
    SAVE_PATH_FIXED + "risk_ladder_final.csv"
)

cluster_means_labeled.to_csv(
    SAVE_PATH_FIXED + "cluster_means_labeled.csv"
)

print("Sauvegarde corrigée minimale terminée")

In [ ]:
with open(
    SAVE_PATH_FIXED + "cluster_label_mappings.json",
    "w"
) as f:
    json.dump(label_mappings_save, f, indent=4)

In [ ]:
SAVE_PATH_FIXED = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels/"

df_check = pd.read_csv(
    SAVE_PATH_FIXED + "final_df_with_kmeans_profiles.csv"
)

print("Profil semaine 3 :")
print(df_check['profile_label_w3'].value_counts())

print("\nProfil semaine 7 :")
print(df_check['profile_label_w7'].value_counts())

print("\nProfil semaine 12 :")
print(df_check['profile_label_w12'].value_counts())

print("\nProfil final :")
print(df_check['profile_label'].value_counts())

In [ ]:
with open(SAVE_PATH_FIXED + "cluster_label_mappings.json", "r") as f:
    mappings_check = json.load(f)

print(mappings_check['3'])
print(mappings_check['risk_order'])

In [ ]:
def build_checkpoint_features(cp):

    tmp = weekly_combined[
        weekly_combined['week'] < cp
    ]

    checkpoint_df = tmp.groupby(student_id_cols).agg(

        active_days_cp = ('weekly_active_days', 'sum'),
        act_types_cp=('weekly_act_types', 'sum'),
        unique_res_cp = ('weekly_unique_res', 'sum'),

        n_submitted_cp = ('weekly_n_submitted', 'sum'),

        n_late_cp = ('weekly_n_late', 'sum'),

        assessment_clicks_cp = (
            'weekly_assessment_clicks',
            'sum'
        ),

        content_clicks_cp = (
            'weekly_content_clicks',
            'sum'
        ),

        social_clicks_cp = (
            'weekly_social_clicks',
            'sum'
        ),

        resource_clicks_cp = (
            'weekly_resource_clicks',
            'sum'
        ),

        specialized_clicks_cp = (
            'weekly_specialized_clicks',
            'sum'
        )

    ).reset_index()

    return checkpoint_df

In [ ]:
checkpoint_features = {}

for cp in [3, 7, 12]:

    checkpoint_features[cp] = (
        build_checkpoint_features(cp)
        .rename(columns=lambda c:
                f"{c}_w{cp}"
                if c not in student_id_cols
                else c
        )
    )

    final_df_wcdmacp = final_df_wcdmacp.merge(
        checkpoint_features[cp],
        on=student_id_cols,
        how='left',
        suffixes=('', f'_w{cp}')
    )

In [ ]:
# ════════════════════════════════════════════════════════════
# METRIQUES SPÉCIFIQUES PAR CHECKPOINT — SANS DATA LEAKAGE
# ════════════════════════════════════════════════════════════

# Features comportementales disponibles par checkpoint
BEHAVIOR_FEATURES_BY_CP = {
    3: [
        'clicks_sum_w3',
        'active_days_cp_w3',
        'score_mean_w3',
        'n_submitted_cp_w3',
        'n_late_cp_w3',
        'assessment_clicks_cp_w3',
        'content_clicks_cp_w3',
        'social_clicks_cp_w3',
        'resource_clicks_cp_w3',
        'specialized_clicks_cp_w3'
    ],
    7: [
        'clicks_sum_w7',
        'active_days_cp_w7',
        'score_mean_w7',
        'n_submitted_cp_w7',
        'n_late_cp_w7',
        'assessment_clicks_cp_w7',
        'content_clicks_cp_w7',
        'social_clicks_cp_w7',
        'resource_clicks_cp_w7',
        'specialized_clicks_cp_w7'
    ],
    12: [
        'clicks_sum_w12',
        'active_days_cp_w12',
        'score_mean_w12',
        'n_submitted_cp_w12',
        'n_late_cp_w12',
        'assessment_clicks_cp_w12',
        'content_clicks_cp_w12',
        'social_clicks_cp_w12',
        'resource_clicks_cp_w12',
        'specialized_clicks_cp_w12'
    ],
}

# Features pour le Risk Ladder par checkpoint
RISK_LADDER_FEATURES_BY_CP = {
    3: {
        'clicks':'clicks_sum_w3',
        'active_days':'active_days_cp_w3',
        'mean_score':'score_mean_w3',
        'submitted':'n_submitted_cp_w3',
        'assessment_clicks':'assessment_clicks_cp_w3',
        'content_clicks':'content_clicks_cp_w3',
        'social_clicks':'social_clicks_cp_w3'

    },
    7: {
        'clicks':'clicks_sum_w7',
        'active_days':'active_days_cp_w7',
        'mean_score':'score_mean_w7',
        'submitted':'n_submitted_cp_w7',
        'assessment_clicks':'assessment_clicks_cp_w7',
        'content_clicks':'content_clicks_cp_w7',
        'social_clicks':'social_clicks_cp_w7'
    },
    12: {
        'clicks':'clicks_sum_w12',
        'active_days':'active_days_cp_w12',
        'mean_score':'score_mean_w12',
        'submitted':'n_submitted_cp_w12',
        'assessment_clicks':'assessment_clicks_cp_w12',
        'content_clicks':'content_clicks_cp_w12',
        'social_clicks':'social_clicks_cp_w12'
    },
}


# ════════════════════════════════════════════════════════════
# ETAPE 3 — ANALYSE SÉQUENCE COMPLÈTE
# Heatmap + Trajectoires + Risk Ladder
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 3 — Analyse séquence complète")
print("=" * 60)

mean_risk = final_df_wcdmacp['at_risk'].mean()

# Profil moyen — séquence complète utilise toutes les features 
cluster_means_full = final_df_wcdmacp.groupby('cluster_full')[
    behavior_features + ['at_risk']
].mean().round(3)
print("\nProfil moyen par cluster :")
display(cluster_means_full)

# Heatmap séquence complète
plot_heatmap(
    normalize_heatmap(cluster_means_full[behavior_features]),
    'Signatures comportementales — Séquence complète\n'
    f'(SEED={best_seeds["full"]}, Sil={metrics_full["sil"]:.4f})'
)

# Risk Ladder séquence complète — métriques finales (pas de fuite)
risk_ladder_full = final_df_wcdmacp.groupby('cluster_full').agg(
    n_students        = ('id_student',                 'count'),
    at_risk_rate      = ('at_risk',                    'mean'),
    mean_score        = ('mean_score',                 'mean'),
    total_clicks      = ('total_clicks',               'mean'),
    submission_rate   = ('submission_rate',            'mean'),
    assessment_clicks = ('clicks_assessment_activity', 'mean'),
    social_clicks     = ('clicks_social_activity',     'mean'),
).round(3).sort_values('at_risk_rate', ascending=False)

print("\nRisk Ladder — Séquence complète :")
display(risk_ladder_full)
plot_risk_ladder(risk_ladder_full, 'cluster_full',
                 'Risk Ladder — Séquence complète', mean_risk)

# Trajectoires temporelles
weekly_with_profile = weekly_combined.merge(
    final_df_wcdmacp[student_id_cols + ['cluster_full']],
    on=student_id_cols, how='left'
)
trajectory = weekly_with_profile.groupby(
    ['week', 'cluster_full']
)['weekly_clicks'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Trajectoires temporelles — Séquence complète',
             fontweight='bold', fontsize=13)

for ax, (start, end, title) in zip(axes, [
    (0, 38, 'Trajectoires complètes (0→38)'),
    (0, 15, 'Zoom premières semaines (0→15)'),
]):
    data_traj = trajectory[trajectory['week'].between(start, end)]
    for cid in range(K_OPTIMAL):
        data = data_traj[data_traj['cluster_full'] == cid]
        ax.plot(data['week'], data['weekly_clicks'],
                'o-', lw=2, ms=4, color=profile_colors[cid],
                label=f'Cluster {cid}')
    for cp in CHECKPOINTS:
        ax.axvline(x=cp, color='gray', ls=':', alpha=0.5, lw=1.5)
    ax.set_xlabel('Semaine')
    ax.set_ylabel('Clics moyens')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 4 — ANALYSE DES CHECKPOINTS
# CORRECTION : utiliser les métriques du checkpoint courant
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ETAPE 4 — Analyse des checkpoints")
print("=" * 60)

risk_checkpoints = {}

for cp in CHECKPOINTS:
    cluster_col = f'cluster_w{cp}'

    print(f"\n{'='*60}")
    print(f"CHECKPOINT — SEMAINE {cp} "
          f"(SEED={best_seeds[str(cp)]} | "
          f"Sil={metrics_cp_km[cp]['sil']:.4f})")
    print('='*60)

    # ── CORRECTION : features spécifiques au checkpoint ───────
    rl_feats = RISK_LADDER_FEATURES_BY_CP[cp]
    clicks_col = rl_feats['clicks']
    score_col  = rl_feats['mean_score']

    # Risk Ladder avec métriques du checkpoint
    agg_dict = {
        'n_students' : ('id_student', 'count'),
        'at_risk_rate': ('at_risk',   'mean'),
        f'clicks_cp{cp}': (clicks_col, 'mean'),
        f'score_cp{cp}' : (score_col,  'mean'),
    }

    risk_cp = final_df_wcdmacp.groupby(cluster_col).agg(
        **agg_dict
    ).round(3).sort_values('at_risk_rate', ascending=False)

    print(f"\nRisk Ladder — semaine {cp} "
          f"(métriques jusqu'à S{cp} uniquement) :")
    display(risk_cp)
    plot_risk_ladder(risk_cp, cluster_col,
                     f'Risk Ladder — Semaine {cp}', mean_risk)

    # ── CORRECTION : heatmap avec features du checkpoint ──────
    cp_behavior_features = [
        f for f in BEHAVIOR_FEATURES_BY_CP[cp]
        if f in final_df_wcdmacp.columns
    ]

    cluster_means_cp = final_df_wcdmacp.groupby(
        cluster_col
    )[cp_behavior_features].mean()

    plot_heatmap(
        normalize_heatmap(cluster_means_cp),
        f'Signatures comportementales — Semaine {cp}\n'
        f'(métriques S1→S{cp} uniquement | '
        f'SEED={best_seeds[str(cp)]}, '
        f'Sil={metrics_cp_km[cp]["sil"]:.4f})',
        ylabel=f'Cluster sem.{cp}'
    )

    risk_checkpoints[cp] = risk_cp

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — LABELLISATION MANUELLE
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 5 — Labellisation manuelle")
print("=" * 60)

print("""
Instructions :
  Regarder les Risk Ladders + Heatmaps de chaque étape
  et adapter les mappings ci-dessous.

Critères :
  Highly Engaged Learner  → at_risk faible + clicks élevés
  Assessment Specialist   → score élevé + clicks modérés
  Mid-Engager             → engagement modéré (S3 uniquement)
  At-Risk Engager         → at_risk intermédiaire
  Resource Skimmer        → at_risk élevé + clicks faibles
""")

CLUSTER_TO_LABEL_FINAL = {
    0: 'Resource Skimmer',
    1: 'Highly Engaged Learner',
    2: 'Assessment Specialist',
    3: 'At-Risk Engager'
}
CLUSTER_TO_LABEL_W3 = {
    0: 'Highly Engaged Learner',
    1: 'Resource Skimmer',
    2: 'At-Risk Engager',
    3: 'Mid-Engager',
}
CLUSTER_TO_LABEL_W7 = {
    0: 'Highly Engaged Learner',
    1: 'Resource Skimmer',
    2: 'At-Risk Engager',
    3: 'Assessment Specialist',
}
CLUSTER_TO_LABEL_W12 = {
    0: 'Resource Skimmer',
    1: 'Assessment Specialist',
    2: 'At-Risk Engager',
    3: 'Highly Engaged Learner',
}

checkpoint_mappings = {
    'full': CLUSTER_TO_LABEL_FINAL,
    3     : CLUSTER_TO_LABEL_W3,
    7     : CLUSTER_TO_LABEL_W7,
    12    : CLUSTER_TO_LABEL_W12,
}

# Appliquer les labels
final_df_wcdmacp['profile_label'] = (
    final_df_wcdmacp['cluster_full'].map(CLUSTER_TO_LABEL_FINAL)
)
for cp in CHECKPOINTS:
    final_df_wcdmacp[f'profile_label_w{cp}'] = (
        final_df_wcdmacp[f'cluster_w{cp}'].map(
            checkpoint_mappings[cp]
        )
    )

print("Labels appliqués :")
for cp in CHECKPOINTS:
    print(f"\n  Semaine {cp} :")
    print(final_df_wcdmacp[f'profile_label_w{cp}'].value_counts()
          .to_string())
print("\n  Final :")
print(final_df_wcdmacp['profile_label'].value_counts().to_string())

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 6 — VALIDATION FINALE DES LABELS
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 6 — Validation finale des labels")
print("=" * 60)

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Assessment Specialist',
    'Highly Engaged Learner'
]

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist' : '#D97706',
    'At-Risk Engager'       : '#2563EB',
    'Resource Skimmer'      : '#DC2626',
}

# Heatmap finale — métriques finales (pas de fuite)
cluster_means_labeled = final_df_wcdmacp.groupby(
    'profile_label'
)[behavior_features].mean()
cluster_means_labeled = cluster_means_labeled.reindex(profile_order)

plot_heatmap(
    normalize_heatmap(cluster_means_labeled),
    'Signatures comportementales — Labels finaux',
    ylabel='Profil'
)

# Risk Ladder final — métriques finales
risk_final = final_df_wcdmacp.groupby('profile_label').agg(
    n_students      = ('id_student',       'count'),
    at_risk_rate    = ('at_risk',          'mean'),
    mean_score      = ('mean_score',       'mean'),
    total_clicks    = ('total_clicks',     'mean'),
    submission_rate = ('submission_rate',  'mean'),
).round(3).sort_values('at_risk_rate', ascending=False)

print("\nRisk Ladder final :")
display(risk_final)
mean_risk = final_df_wcdmacp['at_risk'].mean()
plot_risk_ladder(
    risk_final, 'profile_label',
    'Risk Ladder — Labels finaux', mean_risk
)

# Average Weekly Click Activity
N_WEEKS = 38
final_df_wcdmacp['avg_weekly_click_activity'] = (
    final_df_wcdmacp['total_clicks'] / N_WEEKS
)

avg_weekly_profile = final_df_wcdmacp.groupby('profile_label').agg(
    n_students=('id_student', 'count'),
    avg_weekly_click_activity=('avg_weekly_click_activity', 'mean'),
    total_clicks=('total_clicks', 'mean'),
    at_risk_rate=('at_risk', 'mean')
).reindex(profile_order).round(3)

print("\nAverage Weekly Click Activity par profil :")
display(avg_weekly_profile)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=final_df_wcdmacp,
    x='profile_label',
    y='avg_weekly_click_activity',
    order=profile_order,
    palette=label_colors,
    errorbar=None
)
plt.title('Average Weekly Click Activity par profil', fontweight='bold', fontsize=13)
plt.xlabel('Profil')
plt.ylabel('Average Weekly Click Activity')
plt.xticks(rotation=25, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nValidation finale terminée ")

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 6 — VALIDATION FINALE DES LABELS
# ════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("ÉTAPE 6 — Validation finale des labels")
print("=" * 60)

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Assessment Specialist',
    'Highly Engaged Learner'
]

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist': '#D97706',
    'At-Risk Engager': '#2563EB',
    'Resource Skimmer': '#DC2626',
}

# ── Heatmap finale avec labels ───────────────────────────────
cluster_means_labeled = final_df_wcdmacp.groupby(
    'profile_label'
)[behavior_features].mean()

cluster_means_labeled = cluster_means_labeled.reindex(profile_order)

plot_heatmap(
    normalize_heatmap(cluster_means_labeled),
    'Signatures comportementales — Labels finaux',
    ylabel='Profil'
)

# ── Risk Ladder final avec labels ────────────────────────────
risk_final = final_df_wcdmacp.groupby('profile_label').agg(
    n_students      = ('id_student',       'count'),
    at_risk_rate    = ('at_risk',          'mean'),
    mean_score      = ('mean_score',       'mean'),
    total_clicks    = ('total_clicks',     'mean'),
    submission_rate = ('submission_rate',  'mean'),
).round(3).sort_values('at_risk_rate', ascending=False)

print("\nRisk Ladder final :")
display(risk_final)
mean_risk = final_df_wcdmacp['at_risk'].mean()
plot_risk_ladder(
    risk_final,
    'profile_label',
    'Risk Ladder — Labels finaux',
    mean_risk
)

# ── Average Weekly Click Activity ────────────────────────────

N_WEEKS = 39

final_df_wcdmacp['avg_weekly_click_activity'] = (
    final_df_wcdmacp['total_clicks'] / N_WEEKS
)

avg_weekly_profile = final_df_wcdmacp.groupby('profile_label').agg(
    n_students=('id_student', 'count'),
    avg_weekly_click_activity=('avg_weekly_click_activity', 'mean'),
    total_clicks=('total_clicks', 'mean'),
    at_risk_rate=('at_risk', 'mean')
).reindex(profile_order).round(3)

print("\nAverage Weekly Click Activity par profil :")
display(avg_weekly_profile)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=final_df_wcdmacp,
    x='profile_label',
    y='avg_weekly_click_activity',
    order=profile_order,
    palette=label_colors,
    errorbar=None
)

plt.title(
    'Average Weekly Click Activity par profil',
    fontweight='bold',
    fontsize=13
)
plt.xlabel('Profil')
plt.ylabel('Average Weekly Click Activity')
plt.xticks(rotation=25, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nValidation finale terminée")

In [ ]:

# ════════════════════════════════════════════════════════════
# TRAJECTOIRES TEMPORELLES PROPRES
# ════════════════════════════════════════════════════════════

profile_order = [
    'Resource Skimmer',
    'At-Risk Engager',
    'Mid-Engager',
    'Assessment Specialist',
    'Highly Engaged Learner'
]

label_colors = {
    'Highly Engaged Learner': '#16A34A',
    'Assessment Specialist': '#D97706',
    'Mid-Engager': '#F59E0B',
    'At-Risk Engager': '#2563EB',
    'Resource Skimmer': '#DC2626',
}

profile_df = final_df_wcdmacp[
    student_id_cols + ['profile_label']
].drop_duplicates()

weeks_df  = pd.DataFrame({'week': range(0, 39)})
full_grid = profile_df.merge(weeks_df, how='cross')

weekly_tmp = weekly_combined[
    student_id_cols + ['week', 'weekly_clicks']
].copy()

weekly_full = full_grid.merge(
    weekly_tmp, on=student_id_cols + ['week'], how='left'
)
weekly_full['weekly_clicks'] = weekly_full['weekly_clicks'].fillna(0)

trajectory_labeled = weekly_full.groupby(
    ['week', 'profile_label']
)['weekly_clicks'].mean().reset_index()

trajectory_labeled['weekly_clicks_smooth'] = (
    trajectory_labeled
    .sort_values(['profile_label', 'week'])
    .groupby('profile_label')['weekly_clicks']
    .transform(
        lambda s: s.rolling(window=3, center=True, min_periods=1).mean()
    )
)

plt.figure(figsize=(14, 7))
for label in profile_order:
    if label not in trajectory_labeled['profile_label'].unique():
        continue
    data = trajectory_labeled[
        trajectory_labeled['profile_label'] == label
    ]
    plt.plot(
        data['week'], data['weekly_clicks_smooth'],
        marker='o', linewidth=2.5, markersize=5,
        color=label_colors[label], label=label
    )

milestones = {7: 'Month 1', 14: 'Mid-term', 21: 'Month 3', 35: 'Finals'}
for week, text in milestones.items():
    plt.axvline(x=week, color='gray', linestyle='--', alpha=0.5, lw=1.5)
    plt.text(week + 0.2, plt.ylim()[1] * 0.95, text,
             rotation=45, color='gray', fontsize=9, va='top')

plt.title('Évolution temporelle de l’activité de clics\npar profil final d’étudiant',
          fontweight='bold', fontsize=14)
plt.xlabel('Semaine')
plt.ylabel('Activité moyenne hebdomadaire de clics')
plt.legend(title='Profils d’étudiants')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# ════════════════════════════════════════════════════════════
# VALIDATION DES LABELS — CHECKPOINTS
# heatmap et Risk Ladder avec métriques du checkpoint
# ════════════════════════════════════════════════════════════
label_colors


checkpoint_label_cols = {
    3 : 'profile_label_w3',
    7 : 'profile_label_w7',
    12: 'profile_label_w12',
}

for cp in CHECKPOINTS:
    print("\n" + "=" * 60)
    print(f"VALIDATION LABELS — CHECKPOINT SEMAINE {cp}")
    print("=" * 60)

    label_col = checkpoint_label_cols[cp]

    # ──heatmap avec features du checkpoint ──────
    cp_behavior_features = [
        f for f in BEHAVIOR_FEATURES_BY_CP[cp]
        if f in final_df_wcdmacp.columns
    ]

    cluster_cp = final_df_wcdmacp.groupby(
        label_col
    )[cp_behavior_features].mean()

    present_profiles = [
        p for p in profile_order
        if p in final_df_wcdmacp[label_col].dropna().unique()
    ]
    cluster_cp = cluster_cp.reindex(present_profiles)

    plot_heatmap(
        normalize_heatmap(cluster_cp),
        f'Signatures comportementales — Semaine {cp}',
        ylabel='Profil'
    )

    # ──Risk Ladder avec métriques du checkpoint ─
    rl_feats   = RISK_LADDER_FEATURES_BY_CP[cp]
    clicks_col = rl_feats['clicks']
    score_col  = rl_feats['mean_score']

    risk_cp = final_df_wcdmacp.groupby(label_col).agg(
        n_students             = ('id_student',  'count'),
        at_risk_rate           = ('at_risk',     'mean'),
        **{f'clicks_S{cp}'    : (clicks_col,    'mean')},
        **{f'score_mean_S{cp}': (score_col,     'mean')},
    ).round(3).sort_values('at_risk_rate', ascending=False)

    print(f"\nRisk Ladder — semaine {cp} ")
    display(risk_cp)

    plot_risk_ladder(
        risk_cp, label_col,
        f'Risk Ladder — Semaine {cp}',
        mean_risk
    )

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 7 — P-CEA
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ÉTAPE 7 — P-CEA (Profile Cluster Evolution Analysis)")
print("=" * 60)


profile_steps = [f'profile_label_w{cp}' for cp in CHECKPOINTS] \
                + ['profile_label']
step_names    = [f'Semaine {cp}' for cp in CHECKPOINTS] + ['Final']

pcea_df = final_df_wcdmacp[
    student_id_cols + profile_steps + ['at_risk']
].copy()

for col in profile_steps:
    level_col = col.replace('profile_label', 'risk_level')
    pcea_df[level_col] = pcea_df[col].map(PROFILE_RISK_ORDER)

trajectory_cols = [col.replace('profile_label', 'risk_level')
                   for col in profile_steps]

def compute_trend(row):
    vals = [row[c] for c in trajectory_cols
            if not pd.isna(row.get(c))]
    if len(vals) < 2:
        return np.nan
    return float(np.polyfit(range(len(vals)), vals, 1)[0])

pcea_df['profile_trend'] = pcea_df.apply(compute_trend, axis=1)
pcea_df['trajectory']    = pd.cut(
    pcea_df['profile_trend'],
    bins=[-np.inf, -0.2, 0.2, np.inf],
    labels=['Amélioration', 'Stable', 'Dégradation']
)
pcea_df['trajectory_num'] = pcea_df['trajectory'].map({
    'Amélioration': -1, 'Stable': 0, 'Dégradation': 1
})

cols_to_drop = [
    'profile_trend', 'trajectory', 'trajectory_num',
    'profile_trend_x', 'trajectory_x', 'trajectory_num_x',
    'profile_trend_y', 'trajectory_y', 'trajectory_num_y',
]
final_df_wcdmacp = final_df_wcdmacp.drop(
    columns=[c for c in cols_to_drop
             if c in final_df_wcdmacp.columns]
)

final_df_wcdmacp = final_df_wcdmacp.merge(
    pcea_df[student_id_cols + [
        'profile_trend', 'trajectory', 'trajectory_num'
    ]],
    on=student_id_cols, how='left'
)

print("\nDistribution des trajectoires :")
print(final_df_wcdmacp['trajectory'].value_counts().to_string())

traj_risk = final_df_wcdmacp.groupby(
    'trajectory', observed=True
).agg(
    n_students   = ('at_risk', 'count'),
    at_risk_rate = ('at_risk', 'mean'),
    mean_trend   = ('profile_trend', 'mean')
).round(3)
print("\nRésumé P-CEA :")
display(traj_risk)

In [ ]:
# ── Sankey P-CEA ─────────────────────────────────────────────
labels_sankey   = []
node_colors_sk  = []
for step in step_names:
    for profile in profile_order:
        labels_sankey.append(f"{step}<br>{profile}")
        node_colors_sk.append(label_colors[profile])

label_to_idx = {l: i for i, l in enumerate(labels_sankey)}
sources, targets, values_sk, link_colors_sk = [], [], [], []

for i in range(len(profile_steps) - 1):
    col_from = profile_steps[i]
    col_to   = profile_steps[i+1]
    transitions = pcea_df.groupby(
        [col_from, col_to]
    ).size().reset_index(name='count')

    for _, row in transitions.iterrows():
        src = f"{step_names[i]}<br>{row[col_from]}"
        tgt = f"{step_names[i+1]}<br>{row[col_to]}"
        if src in label_to_idx and tgt in label_to_idx:
            sources.append(label_to_idx[src])
            targets.append(label_to_idx[tgt])
            values_sk.append(row['count'])
            link_colors_sk.append('rgba(150,150,150,0.25)')

fig_sankey = go.Figure(data=[go.Sankey(
    arrangement = "snap",
    node = dict(
        pad=18, thickness=18,
        line=dict(color="black", width=0.3),
        label=labels_sankey, color=node_colors_sk
    ),
    link = dict(
        source=sources, target=targets,
        value=values_sk, color=link_colors_sk
    )
)])
fig_sankey.update_layout(
    title_text = "P-CEA — Migrations des profils — K-Means",
    font_size  = 11, height=650, width=1200
)
fig_sankey.show()


# ── Évolution nombre d'étudiants par profil ────────────────
evolution_counts = [
    {'step': sn, 'profile': p,
     'n_students': pcea_df[col].value_counts().get(p, 0)}
    for sn, col in zip(step_names, profile_steps)
    for p in profile_order
]
evolution_df = pd.DataFrame(evolution_counts)

plt.figure(figsize=(12, 6))
for profile in profile_order:
    data = evolution_df[evolution_df['profile'] == profile]
    plt.plot(data['step'], data['n_students'],
             marker='o', lw=2.5, label=profile,
             color=label_colors[profile])
    for _, row in data.iterrows():
        plt.text(row['step'], row['n_students'] + 200,
                 f"{int(row['n_students'])}", ha='center', fontsize=9)

plt.title("P-CEA — Évolution du nombre d'étudiants par profil",
          fontweight='bold')
plt.xlabel("Étape temporelle")
plt.ylabel("Nombre d'étudiants")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ── Matrices de transition ────────────────────────────────

transition_pairs = [
    (profile_steps[i], profile_steps[i+1],
     f'{step_names[i]} → {step_names[i+1]}')
    for i in range(len(profile_steps) - 1)
]

def get_profiles_present(col):
    return [
        p for p in profile_order
        if p in pcea_df[col].dropna().unique()
    ]

for col_from, col_to, title in transition_pairs:

    row_order = get_profiles_present(col_from)
    col_order = get_profiles_present(col_to)

    matrix = pd.crosstab(
        pcea_df[col_from],
        pcea_df[col_to],
        normalize='index'
    ).reindex(
        index=row_order,
        columns=col_order
    ).fillna(0)

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        matrix,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5,
        cbar_kws={'label': 'Probabilité de transition'}
    )

    plt.title(
        f"P-CEA — Matrice de transition\n{title}",
        fontweight='bold'
    )
    plt.xlabel("Profil destination")
    plt.ylabel("Profil origine")
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# SAUVEGARDE
# ════════════════════════════════════════════════════════════
SAVE_PATH_FIXED = OUTPUT_PATH + "dynamic_profiling_kmeans_fixed_labels3/"
os.makedirs(SAVE_PATH_FIXED, exist_ok=True)

final_df_wcdmacp.to_csv(
    SAVE_PATH_FIXED + "final_df_with_kmeans_profiles.csv",
    index=False
)
pcea_df.to_csv(
    SAVE_PATH_FIXED + "pcea_kmeans_results.csv",
    index=False
)


risk_final.to_csv(SAVE_PATH_FIXED + "risk_ladder_final.csv")
cluster_means_labeled.to_csv(
    SAVE_PATH_FIXED + "cluster_means_labeled.csv"
)

print("Sauvegarde terminée ")

In [ ]:
# Label mappings
label_mappings_save = {
    str(k): {str(ki): vi for ki, vi in v.items()}
    for k, v in checkpoint_mappings.items()
}
label_mappings_save['risk_order'] = PROFILE_RISK_ORDER
label_mappings_save['best_seeds'] = {
    str(k): v for k, v in best_seeds.items()
}

In [ ]:
with open(
    OUTPUT_PATH + "dynamic_profiling_kmeans/cluster_label_mappings.json",
    "r"
) as f:
    old_mapping = json.load(f)

label_mappings_save['best_seeds'] = old_mapping.get(
    'best_seeds', {}
)

In [ ]:
with open(SAVE_PATH_FIXED + "cluster_label_mappings.json", "w") as f:
    json.dump(label_mappings_save, f, indent=4)

In [ ]:
SAVE_PATH = OUTPUT_PATH + "dynamic_profiling_kmeans/"

emb_full_norm = np.load(
    SAVE_PATH + "emb_full_norm.npy"
)

In [ ]:
import pickle

with open(SAVE_PATH + "emb_norm_cp.pkl", "rb") as f:
    emb_norm_cp = pickle.load(f)

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ════════════════════════════════════════════════════════════

# COULEURS DES PROFILS

# ════════════════════════════════════════════════════════════

label_colors = {
'Highly Engaged Learner': '#16A34A',
'Assessment Specialist' : '#D97706',
'Mid-Engager'           : '#F59E0B',
'At-Risk Engager'       : '#2563EB',
'Resource Skimmer'      : '#DC2626',
}

# ════════════════════════════════════════════════════════════

# FONCTION t-SNE

# ════════════════════════════════════════════════════════════

def plot_tsne_profiles( embeddings, labels, title, seed):
  print(f"Calcul t-SNE : {title}")
  tsne = TSNE(
      n_components=2,
      perplexity=30,
      learning_rate='auto',
      init='pca',
      random_state=seed
  )

  emb_2d = tsne.fit_transform(embeddings)

  tsne_df = pd.DataFrame({
      'x': emb_2d[:, 0],
      'y': emb_2d[:, 1],
      'profile': labels
  })

  plt.figure(figsize=(9, 7))

  sns.scatterplot(
      data=tsne_df,
      x='x',
      y='y',
      hue='profile',
      palette=label_colors,
      alpha=0.55,
      s=15,
      linewidth=0
  )

  plt.title(
      title,
      fontweight='bold'
  )

  plt.xlabel('Dimension t-SNE 1')
  plt.ylabel('Dimension t-SNE 2')

  plt.legend(title='Profils',)

  plt.grid(alpha=0.2)
  plt.tight_layout()
  plt.show()

  return emb_2d

# ════════════════════════════════════════════════════════════

# t-SNE FINAL

# ════════════════════════════════════════════════════════════

tsne_full = plot_tsne_profiles(
    embeddings=emb_full_norm,
    labels=final_df_wcdmacp['profile_label'],
    title=(
        't-SNE des profils finaux\n'
        f'Silhouette = {metrics_full["sil"]:.4f}'
    ),
    seed=best_seeds["full"]
    )

# ════════════════════════════════════════════════════════════

# t-SNE SEMAINE 3

# ════════════════════════════════════════════════════════════

tsne_w3 = plot_tsne_profiles(
    embeddings=emb_norm_cp[3],
    labels=final_df_wcdmacp['profile_label_w3'],
    title=(
        't-SNE des profils — Semaine 3\n'
        f'Silhouette = {metrics_cp_km[3]["sil"]:.4f}'
    ),
    seed=best_seeds[str(3)]
    )

# ════════════════════════════════════════════════════════════

# t-SNE SEMAINE 7

# ════════════════════════════════════════════════════════════

tsne_w7 = plot_tsne_profiles(
    embeddings=emb_norm_cp[7],
    labels=final_df_wcdmacp['profile_label_w7'],
    title=(
        't-SNE des profils — Semaine 7\n'
        f'Silhouette = {metrics_cp_km[7]["sil"]:.4f}'
    ),
    seed=best_seeds[str(7)]
)

# ════════════════════════════════════════════════════════════

# t-SNE SEMAINE 12

# ════════════════════════════════════════════════════════════

tsne_w12 = plot_tsne_profiles(
    embeddings=emb_norm_cp[12],
    labels=final_df_wcdmacp['profile_label_w12'],
    title=(
        't-SNE des profils — Semaine 12\n'
        f'Silhouette = {metrics_cp_km[12]["sil"]:.4f}'
    ),
    seed=best_seeds[str(12)]
)


In [ ]:
from sklearn.manifold import TSNE
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, cp in zip(axes, [3, 7, 12]):

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate='auto',
        init='pca',
        random_state=best_seeds[str(cp)]
    )

    emb_2d = tsne.fit_transform(emb_norm_cp[cp])

    tsne_df = pd.DataFrame({
        'x': emb_2d[:, 0],
        'y': emb_2d[:, 1],
        'profile': final_df_wcdmacp[f'profile_label_w{cp}']
    })

    sns.scatterplot(
        data=tsne_df,
        x='x',
        y='y',
        hue='profile',
        palette=label_colors,
        alpha=0.55,
        s=15,
        linewidth=0,
        ax=ax,
        legend=True
    )

    ax.set_title(
        f'Semaine {cp}\nSil={metrics_cp_km[cp]["sil"]:.4f}',
        fontweight='bold'
    )

    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(alpha=0.2)
    ax.legend(title='Profils')

plt.suptitle(
    'Visualisation t-SNE des profils étudiants par checkpoint',
    fontsize=15,
    fontweight='bold'
)

plt.tight_layout()
plt.show()